<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_IRVG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# Cell 1: Setup and Initial Data
# Import necessary libraries
import pandas as pd
import numpy as np

# --- Initial Portfolio Data ---
# This data represents the starting positions and their sensitivities.
data = [
    {'Position ID': 1, 'Currency': 'EUR', 'Type': 'Inflation', 'Option Maturity': 10.0, 'Underlying Maturity': np.nan, 'Sensitivity': 27949},
    {'Position ID': 2, 'Currency': 'EUR', 'Type': 'Inflation', 'Option Maturity': 10.0, 'Underlying Maturity': np.nan, 'Sensitivity': -10962},
    {'Position ID': 3, 'Currency': 'EUR', 'Type': 'Inflation', 'Option Maturity': 0.5, 'Underlying Maturity': np.nan, 'Sensitivity': -9063},
    {'Position ID': 4, 'Currency': 'USD', 'Type': 'Risk-Free', 'Option Maturity': 0.5, 'Underlying Maturity': 1.0, 'Sensitivity': -5328},
    {'Position ID': 5, 'Currency': 'USD', 'Type': 'Risk-Free', 'Option Maturity': 0.5, 'Underlying Maturity': 3.0, 'Sensitivity': -5477}
]

# --- Regulatory Parameters ---
# Risk Weight for GIRR Vega as per Article 325ax
risk_weight_girr = 1.00  # 100%

# Cross-Bucket Correlation for GIRR as per Article 325ag
gamma_bc_medium = 0.50 # 50%

# Create an initial DataFrame
portfolio_df = pd.DataFrame(data)

print("--- Initial Setup Complete ---")
print(f"Loaded {len(portfolio_df)} positions.")

--- Initial Setup Complete ---
Loaded 5 positions.


In [23]:
# Cell 2: Step 1 & 2 - Identify Risk Factors and Sensitivities
# The risk factor is a combination of Currency, Option Maturity, and Underlying Maturity.

print("\n### Step 1 & 2: Identify Risk Factors and Sensitivities ###")
print("The initial portfolio with sensitivities for each position:")

# Display the initial portfolio DataFrame
display(portfolio_df.set_index('Position ID'))


### Step 1 & 2: Identify Risk Factors and Sensitivities ###
The initial portfolio with sensitivities for each position:


,Currency,Type,Option Maturity,Underlying Maturity,Sensitivity
Position ID,,,,,
1,EUR,Inflation,10.0,NaN,27949
2,EUR,Inflation,10.0,NaN,-10962
3,EUR,Inflation,0.5,NaN,-9063
4,USD,Risk-Free,0.5,1.0,-5328
5,USD,Risk-Free,0.5,3.0,-5477


In [24]:
# Cell 3: Step 3 - Calculate Net Sensitivities (sk)
# As per Article 325f(5), we net sensitivities for identical risk factors.

print("\n### Step 3: Calculate Net Sensitivities (sk) ###")
print("Sensitivities are grouped by their unique risk factor and summed.")

# Define the risk factor components. For GIRR Vega, this is Currency, Option Maturity, and Underlying Maturity.
# The 'Type' column is descriptive and not part of the risk factor definition for netting.
risk_factor_cols = ['Currency', 'Option Maturity', 'Underlying Maturity']

# Group by risk factors and sum sensitivities to get the net sensitivity
net_sensitivities_df = portfolio_df.groupby(risk_factor_cols, dropna=False).agg(
    Net_Sensitivity_sk=('Sensitivity', 'sum')
).reset_index()

# Add a Risk Factor ID for clarity
net_sensitivities_df['Risk_Factor_ID'] = [f"RF{i+1}" for i in range(len(net_sensitivities_df))]
net_sensitivities_df.set_index('Risk_Factor_ID', inplace=True)

display(net_sensitivities_df)


### Step 3: Calculate Net Sensitivities (sk) ###
Sensitivities are grouped by their unique risk factor and summed.


,Currency,Option Maturity,Underlying Maturity,Net_Sensitivity_sk
Risk_Factor_ID,,,,
RF1,EUR,0.5,NaN,-9063
RF2,EUR,10.0,NaN,16987
RF3,USD,0.5,1.0,-5328
RF4,USD,0.5,3.0,-5477


In [25]:
# Cell 4: Step 4 - Calculate Weighted Sensitivities (WSk)
# Each net sensitivity is multiplied by its regulatory risk weight (RWk).

print("\n### Step 4: Calculate Weighted Sensitivities (WSk) ###")
print("Applying regulatory risk weights to the net sensitivities.")

net_sensitivities_df['Risk_Weight_RWk'] = risk_weight_girr
net_sensitivities_df['Weighted_Sensitivity_WSk'] = net_sensitivities_df['Net_Sensitivity_sk'] * net_sensitivities_df['Risk_Weight_RWk']

display(net_sensitivities_df[['Currency', 'Net_Sensitivity_sk', 'Risk_Weight_RWk', 'Weighted_Sensitivity_WSk']])


### Step 4: Calculate Weighted Sensitivities (WSk) ###
Applying regulatory risk weights to the net sensitivities.


,Currency,Net_Sensitivity_sk,Risk_Weight_RWk,Weighted_Sensitivity_WSk
Risk_Factor_ID,,,,
RF1,EUR,-9063,1.0,-9063.0
RF2,EUR,16987,1.0,16987.0
RF3,USD,-5328,1.0,-5328.0
RF4,USD,-5477,1.0,-5477.0


In [26]:
# Cell 5: Step 5 & 6 - Determine Correlation Parameters (Medium Scenario)
# Calculate intra-bucket correlations (rho_kl) and identify cross-bucket correlation (gamma_bc).

print("\n### Step 5 & 6: Determine Correlation Parameters (Medium Scenario) ###")

# For perfect replication of the manual example's methodology, we use the rounded values.
rho_eur_medium = 0.827
rho_usd_medium = 0.980

print("Intra-Bucket Correlations (rho_kl):")
print(f"  - EUR Bucket: {rho_eur_medium:.1%}")
print(f"  - USD Bucket: {rho_usd_medium:.1%}")
print("\nCross-Bucket Correlation (gamma_bc):")
print(f"  - EUR vs USD: {gamma_bc_medium:.1%}")


### Step 5 & 6: Determine Correlation Parameters (Medium Scenario) ###
Intra-Bucket Correlations (rho_kl):
  - EUR Bucket: 82.7%
  - USD Bucket: 98.0%

Cross-Bucket Correlation (gamma_bc):
  - EUR vs USD: 50.0%


In [27]:
# Cell 6: Step 7 - Intra-Bucket Aggregation (Medium Scenario)
# This function calculates the bucket-specific capital (Kb).

def calculate_kb(ws_vector, rho):
    """Calculates bucket-specific capital (Kb)."""
    ws_vector = np.array(ws_vector)
    sum_ws_sq = np.sum(ws_vector**2)

    cross_product_sum = 0
    if len(ws_vector) > 1:
        # Sum of (rho_kl * WSk * WSl) for all pairs where k != l
        # For two factors, this is 2 * rho * WS1 * WS2
        cross_product_sum = 2 * rho * ws_vector[0] * ws_vector[1]

    k_squared = max(sum_ws_sq + cross_product_sum, 0)
    return np.sqrt(k_squared)

print("\n### Step 7: Intra-Bucket Aggregation (Medium Scenario) ###")
print("Calculating bucket-specific capital (Kb).")

# Calculate Kb for each bucket
ws_eur = net_sensitivities_df.loc[net_sensitivities_df['Currency'] == 'EUR', 'Weighted_Sensitivity_WSk'].values
kb_eur_medium = calculate_kb(ws_eur, rho_eur_medium)

ws_usd = net_sensitivities_df.loc[net_sensitivities_df['Currency'] == 'USD', 'Weighted_Sensitivity_WSk'].values
kb_usd_medium = calculate_kb(ws_usd, rho_usd_medium)

# Calculate Sb for each bucket (sum of weighted sensitivities)
sb_eur = ws_eur.sum()
sb_usd = ws_usd.sum()

kb_results_df = pd.DataFrame({
    'Bucket': ['EUR', 'USD'],
    'Kb_Medium': [kb_eur_medium, kb_usd_medium],
    'Sb_Medium': [sb_eur, sb_usd]
}).set_index('Bucket')

display(kb_results_df.style.format({'Kb_Medium': '{:,.0f}', 'Sb_Medium': '{:,.0f}'}))


### Step 7: Intra-Bucket Aggregation (Medium Scenario) ###
Calculating bucket-specific capital (Kb).


,Kb_Medium,Sb_Medium
Bucket,,
EUR,"10,773","7,924"
USD,"10,751","-10,805"


In [28]:
# Cell 7: Step 8 - Cross-Bucket Aggregation (Medium Scenario)
# Aggregates the bucket-level capital (Kb) to get the final risk class capital.

def calculate_final_capital(kb_series, sb_series, gamma):
    """Calculates the final risk class capital."""
    sum_kb_sq = np.sum(kb_series**2)

    cross_bucket_sum = 0
    if len(sb_series) > 1:
        s_values = sb_series.values
        # For two buckets, this is 2 * gamma * S_b * S_c
        cross_bucket_sum = 2 * gamma * s_values[0] * s_values[1]

    final_capital_sq = max(sum_kb_sq + cross_bucket_sum, 0)
    return np.sqrt(final_capital_sq)

print("\n### Step 8: Cross-Bucket Aggregation (Medium Scenario) ###")
print("Aggregating bucket capital using cross-bucket correlations.")

medium_scenario_capital = calculate_final_capital(kb_results_df['Kb_Medium'], kb_results_df['Sb_Medium'], gamma_bc_medium)
print(f"\nMedium Scenario GIRR Vega Capital Requirement: {medium_scenario_capital:,.0f} EUR")


### Step 8: Cross-Bucket Aggregation (Medium Scenario) ###
Aggregating bucket capital using cross-bucket correlations.

Medium Scenario GIRR Vega Capital Requirement: 12,084 EUR


In [29]:
# Cell 8: Step 9 - Calculate High and Low Correlation Scenarios
# Per Article 325h, the calculation is repeated for high and low correlation scenarios.

print("\n### Step 9: Calculate High and Low Correlation Scenarios ###")

# --- High Correlation Scenario ---
rho_eur_high = min(1.25 * rho_eur_medium, 1.0)
rho_usd_high = min(1.25 * rho_usd_medium, 1.0)
gamma_bc_high = min(1.25 * gamma_bc_medium, 1.0)

# Recalculate Kb for the High scenario
kb_eur_high = calculate_kb(ws_eur, rho_eur_high)
kb_usd_high = calculate_kb(ws_usd, rho_usd_high)
high_scenario_capital = calculate_final_capital(pd.Series([kb_eur_high, kb_usd_high]), pd.Series([sb_eur, sb_usd]), gamma_bc_high)

# --- Low Correlation Scenario ---
rho_eur_low = max(2 * rho_eur_medium - 1, 0.75 * rho_eur_medium)
rho_usd_low = max(2 * rho_usd_medium - 1, 0.75 * rho_usd_medium)
gamma_bc_low = max(2 * gamma_bc_medium - 1, 0.75 * gamma_bc_medium)

# Recalculate Kb for the Low scenario
kb_eur_low = calculate_kb(ws_eur, rho_eur_low)
kb_usd_low = calculate_kb(ws_usd, rho_usd_low)
low_scenario_capital = calculate_final_capital(pd.Series([kb_eur_low, kb_usd_low]), pd.Series([sb_eur, sb_usd]), gamma_bc_low)

# --- Display Summary ---
scenario_summary = pd.DataFrame([
    {'Scenario': 'Medium', 'Capital': medium_scenario_capital},
    {'Scenario': 'High', 'Capital': high_scenario_capital},
    {'Scenario': 'Low', 'Capital': low_scenario_capital}
])
print("\n--- Summary of Capital Charges by Scenario ---")
display(scenario_summary.set_index('Scenario').style.format({'Capital': '{:,.0f}'}))


### Step 9: Calculate High and Low Correlation Scenarios ###

--- Summary of Capital Charges by Scenario ---


,Capital
Scenario,
Medium,"12,084"
High,"8,516"
Low,"14,816"


In [30]:
# Cell 9: Step 10 - Final Charge Calculation
# The final own funds requirement is the maximum of the three scenarios.

print("\n### Step 10: Final Charge Calculation ###")

final_capital_charge = scenario_summary['Capital'].max()

print(f"\nFinal GIRR Vega Capital Requirement: {final_capital_charge:,.0f} EUR")

# Display the final summary card as a DataFrame
summary_card = pd.DataFrame([{'Final GIRR Vega Capital Requirement': f"{final_capital_charge:,.0f} EUR"}])
display(summary_card)


### Step 10: Final Charge Calculation ###

Final GIRR Vega Capital Requirement: 14,816 EUR


,Final GIRR Vega Capital Requirement
0,"14,816 EUR"
